# SentinelIQ — RAG Pipeline Exploration

This notebook walks through the Retrieval-Augmented Generation (RAG) pipeline used by SentinelIQ to give the LangGraph investigation agent access to historical fraud case knowledge.

**What we cover:**
1. Embedding model setup (Ollama `nomic-embed-text`)
2. Upserting historical cases into ChromaDB
3. Querying the vector store with natural language
4. Inspecting similarity scores and metadata
5. Testing the full retrieval chain used by the agent

> **Prerequisites:** Run `python scripts/generate_data.py` first to create `data/synthetic/historical_cases.json`.

In [ ]:
import sys
import os

# Add project root to path so src.* imports work from the notebook
sys.path.insert(0, os.path.abspath('..'))

import json
import pandas as pd
from dotenv import load_dotenv

load_dotenv('../.env')
print('Environment loaded.')

## 1. Inspect the Historical Cases Dataset

These are the 200 synthetic investigator reports that seed the RAG knowledge base.

In [ ]:
with open('../data/synthetic/historical_cases.json', 'r') as f:
    cases = json.load(f)

print(f'Total historical cases: {len(cases)}')
print('\nSample case:')
print(json.dumps(cases[0], indent=2))

In [ ]:
# Distribution of fraud types in the knowledge base
df_cases = pd.DataFrame(cases)
print('Fraud type distribution:')
print(df_cases['fraud_type'].value_counts())
print('\nOutcome distribution:')
print(df_cases['outcome'].value_counts())

## 2. Embedding Model Setup

SentinelIQ uses Ollama's `nomic-embed-text` model for local, on-premise embeddings.
No data leaves the network.

> Make sure Ollama is running: `ollama serve`

In [ ]:
from src.rag.embeddings import get_configured_embeddings

embedder = get_configured_embeddings()

# Test a single embedding
test_text = 'Account with high velocity and IP country mismatch'
embedding = embedder.embed_query(test_text)

print(f'Embedding model: nomic-embed-text')
print(f'Embedding dimensions: {len(embedding)}')
print(f'First 5 values: {embedding[:5]}')

## 3. Upsert Historical Cases into ChromaDB

This step embeds all 200 historical cases and stores them in the persistent ChromaDB vector store.

In [ ]:
from src.rag.vector_store import upsert_historical_cases, get_vector_store

# Upsert all historical cases
upsert_historical_cases('../data/synthetic/historical_cases.json')

# Verify the collection
_, collection = get_vector_store()
count = collection.count()
print(f'\nDocuments in ChromaDB collection: {count}')

## 4. Querying the Vector Store

Let's test similarity search with different query types and inspect the results.

In [ ]:
from src.rag.retriever import retrieve_similar_cases

# Query 1: ATO-style query
query_ato = 'Account exhibited rapid transaction velocity from a foreign IP. New device detected.'
results_ato = retrieve_similar_cases(query_ato, k=3)

print('=== ATO Query Results ===')
for r in results_ato:
    print(f"  Case: {r['case_id']} | Type: {r['fraud_type']} | Similarity: {r['similarity_score']}")
    print(f"  Summary: {r['summary'][:100]}...")
    print()

In [ ]:
# Query 2: Synthetic identity query
query_synth = 'New account shares device with multiple other recently created accounts.'
results_synth = retrieve_similar_cases(query_synth, k=3)

print('=== Synthetic Identity Query Results ===')
for r in results_synth:
    print(f"  Case: {r['case_id']} | Type: {r['fraud_type']} | Similarity: {r['similarity_score']}")
    print(f"  Summary: {r['summary'][:100]}...")
    print()

In [ ]:
# Query 3: With metadata filter — only retrieve confirmed fraud cases
query_filtered = 'High value transaction with failed login attempts'
results_filtered = retrieve_similar_cases(query_filtered, k=3, filter_outcome='confirmed_fraud')

print('=== Filtered Query (confirmed_fraud only) ===')
for r in results_filtered:
    print(f"  Case: {r['case_id']} | Outcome: {r['outcome']} | Similarity: {r['similarity_score']}")
    print()

## 5. Similarity Score Analysis

Let's understand how similarity scores distribute across different query types.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Run multiple queries and collect similarity scores
test_queries = [
    ('ATO', 'Rapid velocity from foreign IP, new device, failed logins'),
    ('Synthetic ID', 'New account shares device with multiple accounts in a ring'),
    ('Card Testing', 'Multiple micro-transactions followed by high value attempt'),
    ('Unrelated', 'Weather forecast for tomorrow in London'),  # Should have low similarity
]

fig, axes = plt.subplots(1, len(test_queries), figsize=(16, 4))
fig.suptitle('RAG Similarity Score Distribution by Query Type', fontsize=14)

for ax, (label, query) in zip(axes, test_queries):
    results = retrieve_similar_cases(query, k=10)
    scores = [r['similarity_score'] for r in results]
    
    ax.bar(range(len(scores)), scores, color='steelblue')
    ax.set_title(label)
    ax.set_xlabel('Rank')
    ax.set_ylabel('Similarity Score')
    ax.set_ylim(0, 1)
    ax.axhline(y=0.7, color='red', linestyle='--', alpha=0.5, label='0.7 threshold')

plt.tight_layout()
plt.savefig('../data/processed/rag_similarity_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to data/processed/rag_similarity_analysis.png')

## 6. Full RAG Chain Test

Test the complete retrieval + LLM generation chain as used by the agent.

In [ ]:
from src.rag.prompts import INVESTIGATION_SYSTEM_PROMPT, build_investigation_prompt
from src.llm.llm_client import get_llm
from langchain_core.messages import SystemMessage, HumanMessage

# Simulate a flagged event
test_event = {
    'transaction_amount': 3200.0,
    'account_age_days': 8,
    'ip_country_mismatch': 1,
    'device_change_count': 2,
    'velocity_1hr': 7,
    'avg_txn_amount_30d': 120.0,
    'failed_login_count_24hr': 3,
}

test_score = {
    'risk_score': 0.91,
    'risk_level': 'CRITICAL',
    'explanation': 'velocity_1hr (+0.41), ip_country_mismatch (+0.28)',
}

# Retrieve similar cases
retrieved = retrieve_similar_cases(test_score['explanation'], k=3)

# Build the prompt
user_prompt = build_investigation_prompt(test_event, test_score, retrieved)
print('=== Investigation Prompt Preview ===')
print(user_prompt[:800] + '...')

In [ ]:
# Call the LLM
llm = get_llm()
messages = [
    SystemMessage(content=INVESTIGATION_SYSTEM_PROMPT),
    HumanMessage(content=user_prompt)
]

print('Calling LLM (this may take 10-30 seconds)...')
response = llm.invoke(messages)

print('\n=== LLM Response ===')
print(response.content)

## 7. Feedback Loop Simulation

Simulate a human reviewer decision being fed back into the knowledge base.

In [ ]:
from src.review.queue import ReviewQueueManager
from src.review.feedback import ReviewFeedbackHandler

# Add a mock case to the queue
mock_case = {
    'case_id': 'NB-TEST-001',
    'fraud_type': 'Account Takeover',
    'confidence': 0.91,
    'evidence_summary': 'High velocity from foreign IP with new device and failed logins.',
    'recommended_action': 'Block transaction and freeze account.',
    'similar_cases': [],
    'reviewer_notes': ''
}

qm = ReviewQueueManager()
qm.add_case(mock_case)
print(f'Cases in queue: {len(qm.get_pending_cases())}')

# Simulate reviewer decision
handler = ReviewFeedbackHandler()
success = handler.log_decision({
    'case_id': 'NB-TEST-001',
    'decision': 'approve',
    'reviewer_notes': 'Confirmed ATO. Customer notified. Account frozen.',
    'reviewer_id': 'notebook_analyst'
})

print(f'Feedback logged: {success}')
print(f'Cases remaining in queue: {len(qm.get_pending_cases())}')

# Verify it's now retrievable
new_results = retrieve_similar_cases('Account Takeover confirmed ATO frozen', k=1)
print(f'\nRetrieved after feedback: {new_results[0]["case_id"] if new_results else "None"}')

## Summary

The RAG pipeline:
- Uses `nomic-embed-text` (local, via Ollama) to embed both documents and queries
- Stores embeddings in ChromaDB with cosine similarity
- Retrieves top-k similar historical cases at investigation time
- Feeds reviewer decisions back into the knowledge base, closing the learning loop

The similarity scores show strong discrimination between relevant and irrelevant queries, confirming the embedding model is well-suited to the fraud domain.